# LangGraph - Multi-Agent и Subgraphs

## Что изучим:

1. **Multi-Agent** - несколько агентов работают вместе
2. **Supervisor Pattern** - один агент управляет другими
3. **Subgraphs** - модульная архитектура графов

### Зачем это нужно?

**Проблема одного агента:**
- Слишком много инструментов — путается
- Сложные задачи — теряет контекст
- Разные роли — конфликт промптов

**Решение — команда агентов:**
- Каждый агент — специалист в своей области
- Supervisor распределяет задачи
- Модульность — легко добавлять/убирать агентов

### Java аналогия:

Multi-agent = Микросервисная архитектура:
- Supervisor = API Gateway / Orchestrator
- Агенты = Отдельные сервисы
- Subgraphs = Модули / Bounded Contexts

In [1]:
# Установка
!pip install -q langgraph langchain langchain-openai python-dotenv

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("DEEPSEEK_API_KEY")
print(f"API Key: {api_key[:10]}..." if api_key else "API Key не найден!")

API Key: sk-***...


In [3]:
from typing import TypedDict, Annotated, Literal
from langgraph.graph import StateGraph, END, START
from langgraph.types import Send
from langgraph.checkpoint.memory import MemorySaver
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
import operator

# LLM
llm = ChatOpenAI(
    base_url="https://api.deepseek.com",
    api_key=api_key,
    model="deepseek-chat",
    temperature=0
)

print("Imports готовы!")

Imports готовы!


---

## 1. Простой Multi-Agent: Handoff Pattern

### Идея:

Агенты передают управление друг другу напрямую.

```
User → Agent A → Agent B → Agent A → Response
```

### Пример: Исследователь + Писатель

- **Researcher** — ищет информацию
- **Writer** — пишет текст на основе найденного

In [ ]:
# Состояние для команды агентов
class TeamState(TypedDict):
    messages: Annotated[list, operator.add]  # История сообщений
    next_agent: str  # Кто следующий
    research_data: str  # Данные от researcher
    final_output: str  # Финальный результат

print("State определён!")

In [ ]:
# Агент-исследователь
def researcher_agent(state: TeamState) -> TeamState:
    print("\n🔍 RESEARCHER работает...")
    
    messages = [
        SystemMessage(content="""Ты — исследователь. Твоя задача:
1. Проанализировать запрос пользователя
2. Найти/сгенерировать ключевые факты по теме
3. Вернуть структурированные данные для писателя

Формат ответа:
ФАКТЫ:
- факт 1
- факт 2
- факт 3
"""),
        HumanMessage(content=state["messages"][-1].content if state["messages"] else "")
    ]
    
    response = llm.invoke(messages)
    print(f"   Найдено: {response.content[:100]}...")
    
    return {
        "messages": [AIMessage(content=f"[Researcher]: {response.content}")],
        "research_data": response.content,
        "next_agent": "writer"
    }

print("Researcher создан!")

In [ ]:
# Агент-писатель
def writer_agent(state: TeamState) -> TeamState:
    print("\n✍️ WRITER работает...")
    
    messages = [
        SystemMessage(content="""Ты — профессиональный писатель. Твоя задача:
1. Взять данные от исследователя
2. Написать engaging текст на их основе
3. Сделать текст интересным и читаемым

Пиши кратко, по делу, с примерами.
"""),
        HumanMessage(content=f"""Данные от исследователя:
{state['research_data']}

Напиши на основе этого короткую статью (3-4 абзаца).""")
    ]
    
    response = llm.invoke(messages)
    print(f"   Написано: {response.content[:100]}...")
    
    return {
        "messages": [AIMessage(content=f"[Writer]: {response.content}")],
        "final_output": response.content,
        "next_agent": "end"
    }

print("Writer создан!")

In [ ]:
# Роутер — кто следующий?
def router(state: TeamState) -> Literal["researcher", "writer", "__end__"]:
    next_agent = state.get("next_agent", "researcher")
    
    if next_agent == "end":
        return "__end__"
    return next_agent

print("Router создан!")

In [ ]:
# Собираем граф
workflow = StateGraph(TeamState)

# Добавляем узлы
workflow.add_node("researcher", researcher_agent)
workflow.add_node("writer", writer_agent)

# Входная точка — сразу идём в router
workflow.add_conditional_edges(
    START,
    router,
    {
        "researcher": "researcher",
        "writer": "writer",
        "__end__": END
    }
)

# После каждого агента — снова в router
workflow.add_conditional_edges("researcher", router)
workflow.add_conditional_edges("writer", router)

# Компилируем
app = workflow.compile()

print("Граф собран!")

In [ ]:
# Визуализация
from IPython.display import Image, display

try:
    display(Image(app.get_graph().draw_mermaid_png()))
except:
    print(app.get_graph().draw_mermaid())

In [ ]:
# Запускаем!
result = app.invoke({
    "messages": [HumanMessage(content="Расскажи про квантовые компьютеры")],
    "next_agent": "researcher",
    "research_data": "",
    "final_output": ""
})

print("\n" + "="*50)
print("ФИНАЛЬНЫЙ РЕЗУЛЬТАТ:")
print("="*50)
print(result["final_output"])

### Что произошло:

1. User → **Researcher** (собрал факты)
2. Researcher → **Writer** (написал статью)
3. Writer → **END** (готово)

### Проблемы этого подхода:

- Жёсткий порядок: researcher → writer
- Нет гибкости в маршрутизации
- Сложно добавлять новых агентов

**Решение → Supervisor Pattern!**

---

## 2. Supervisor Pattern

### Идея:

Один "босс" решает, кого вызвать следующим.

```
User → Supervisor → Agent A
                 → Agent B  
                 → Agent C
                 → Response
```

### Java аналогия:

Supervisor = Orchestrator / Saga Coordinator
- Знает о всех сервисах
- Решает порядок вызовов
- Обрабатывает ошибки

In [ ]:
from langchain_core.tools import tool

# Определяем "работников"
WORKERS = ["researcher", "writer", "critic"]

# Состояние с supervisor
class SupervisorState(TypedDict):
    messages: Annotated[list, operator.add]
    next_worker: str
    research_data: str
    draft: str
    feedback: str

print("State для Supervisor готов!")

In [ ]:
# Supervisor — решает кого вызвать
def supervisor_node(state: SupervisorState) -> SupervisorState:
    print("\n👔 SUPERVISOR думает...")
    
    # Собираем контекст
    context = f"""
Текущее состояние:
- Research данные: {'есть' if state.get('research_data') else 'нет'}
- Черновик: {'есть' if state.get('draft') else 'нет'}
- Фидбек: {'есть' if state.get('feedback') else 'нет'}

Последние сообщения:
{[m.content[:100] for m in state['messages'][-3:]]}
"""
    
    messages = [
        SystemMessage(content=f"""Ты — supervisor команды из 3 работников:
- researcher: ищет информацию, собирает факты
- writer: пишет тексты на основе данных
- critic: проверяет качество, даёт фидбек

Твоя задача — решить, кого вызвать следующим.

Правила:
1. Сначала researcher (если нет данных)
2. Потом writer (если нет черновика)
3. Потом critic (если нет фидбека)
4. Если всё готово — FINISH

Ответь ОДНИМ словом: researcher, writer, critic, или FINISH
"""),
        HumanMessage(content=context)
    ]
    
    response = llm.invoke(messages)
    next_worker = response.content.strip().lower()
    
    # Валидация
    if next_worker not in WORKERS + ["finish"]:
        # Fallback логика
        if not state.get('research_data'):
            next_worker = "researcher"
        elif not state.get('draft'):
            next_worker = "writer"
        elif not state.get('feedback'):
            next_worker = "critic"
        else:
            next_worker = "finish"
    
    print(f"   Решение: {next_worker}")
    
    return {
        "messages": [AIMessage(content=f"[Supervisor]: Вызываю {next_worker}")],
        "next_worker": next_worker
    }

print("Supervisor создан!")

In [ ]:
# Работники
def researcher_node(state: SupervisorState) -> SupervisorState:
    print("\n🔍 RESEARCHER работает...")
    
    task = state["messages"][0].content  # Оригинальный запрос
    
    messages = [
        SystemMessage(content="Ты исследователь. Найди 3-5 ключевых фактов по теме. Кратко, по пунктам."),
        HumanMessage(content=task)
    ]
    
    response = llm.invoke(messages)
    
    return {
        "messages": [AIMessage(content=f"[Researcher]: {response.content}")],
        "research_data": response.content
    }

def writer_node(state: SupervisorState) -> SupervisorState:
    print("\n✍️ WRITER работает...")
    
    messages = [
        SystemMessage(content="Ты писатель. Напиши короткую статью (2-3 абзаца) на основе данных."),
        HumanMessage(content=f"Данные:\n{state['research_data']}")
    ]
    
    response = llm.invoke(messages)
    
    return {
        "messages": [AIMessage(content=f"[Writer]: {response.content}")],
        "draft": response.content
    }

def critic_node(state: SupervisorState) -> SupervisorState:
    print("\n🧐 CRITIC работает...")
    
    messages = [
        SystemMessage(content="""Ты критик. Оцени текст:
1. Что хорошо?
2. Что можно улучшить?
3. Общая оценка (1-10)

Будь конструктивен и краток."""),
        HumanMessage(content=f"Текст для проверки:\n{state['draft']}")
    ]
    
    response = llm.invoke(messages)
    
    return {
        "messages": [AIMessage(content=f"[Critic]: {response.content}")],
        "feedback": response.content
    }

print("Все работники созданы!")

In [ ]:
# Роутер от supervisor
def supervisor_router(state: SupervisorState) -> str:
    next_worker = state.get("next_worker", "researcher")
    
    if next_worker == "finish":
        return "__end__"
    return next_worker

print("Router готов!")

In [ ]:
# Собираем граф
workflow = StateGraph(SupervisorState)

# Узлы
workflow.add_node("supervisor", supervisor_node)
workflow.add_node("researcher", researcher_node)
workflow.add_node("writer", writer_node)
workflow.add_node("critic", critic_node)

# Входная точка — supervisor
workflow.set_entry_point("supervisor")

# Supervisor решает куда идти
workflow.add_conditional_edges(
    "supervisor",
    supervisor_router,
    {
        "researcher": "researcher",
        "writer": "writer",
        "critic": "critic",
        "__end__": END
    }
)

# После каждого работника — обратно к supervisor
workflow.add_edge("researcher", "supervisor")
workflow.add_edge("writer", "supervisor")
workflow.add_edge("critic", "supervisor")

# Компилируем
supervisor_app = workflow.compile()

print("Supervisor граф готов!")

In [ ]:
# Визуализация
try:
    display(Image(supervisor_app.get_graph().draw_mermaid_png()))
except:
    print(supervisor_app.get_graph().draw_mermaid())

In [ ]:
# Запускаем!
result = supervisor_app.invoke({
    "messages": [HumanMessage(content="Напиши про искусственный интеллект в медицине")],
    "next_worker": "",
    "research_data": "",
    "draft": "",
    "feedback": ""
})

print("\n" + "="*50)
print("РЕЗУЛЬТАТЫ:")
print("="*50)
print("\n📝 ЧЕРНОВИК:")
print(result["draft"])
print("\n💬 ФИДБЕК:")
print(result["feedback"])

### Преимущества Supervisor Pattern:

| Аспект | Handoff | Supervisor |
|--------|---------|------------|
| Гибкость | Жёсткий порядок | Динамический |
| Масштабирование | Сложно | Легко добавить агента |
| Контроль | Распределённый | Централизованный |
| Отладка | Сложнее | Проще (всё через boss) |

---

## 3. Subgraphs — Модульная архитектура

### Идея:

Граф внутри графа. Каждый subgraph — отдельный модуль со своей логикой.

```
Main Graph
├── Subgraph A (Research Team)
│   ├── Web Searcher
│   └── Summarizer
└── Subgraph B (Writing Team)
    ├── Drafter
    └── Editor
```

### Java аналогия:

Subgraph = Maven Module / Gradle Subproject:
- Изолированная логика
- Своя область ответственности
- Можно переиспользовать

In [ ]:
# === SUBGRAPH 1: Research Team ===

class ResearchState(TypedDict):
    query: str
    raw_data: str
    summary: str

def search_node(state: ResearchState) -> ResearchState:
    print("   🔎 Searcher: ищу информацию...")
    
    # Симуляция поиска
    response = llm.invoke([
        SystemMessage(content="Сгенерируй 5 фактов по теме. Формат: пункты."),
        HumanMessage(content=state["query"])
    ])
    
    return {"raw_data": response.content}

def summarize_node(state: ResearchState) -> ResearchState:
    print("   📋 Summarizer: обобщаю...")
    
    response = llm.invoke([
        SystemMessage(content="Сделай краткое саммари (2-3 предложения)."),
        HumanMessage(content=state["raw_data"])
    ])
    
    return {"summary": response.content}

# Собираем subgraph
research_workflow = StateGraph(ResearchState)
research_workflow.add_node("search", search_node)
research_workflow.add_node("summarize", summarize_node)
research_workflow.set_entry_point("search")
research_workflow.add_edge("search", "summarize")
research_workflow.add_edge("summarize", END)

research_subgraph = research_workflow.compile()

print("Research Subgraph готов!")

In [ ]:
# === SUBGRAPH 2: Writing Team ===

class WritingState(TypedDict):
    source_material: str
    draft: str
    final_text: str

def draft_node(state: WritingState) -> WritingState:
    print("   ✏️ Drafter: пишу черновик...")
    
    response = llm.invoke([
        SystemMessage(content="Напиши черновик статьи (2 абзаца) на основе материала."),
        HumanMessage(content=state["source_material"])
    ])
    
    return {"draft": response.content}

def edit_node(state: WritingState) -> WritingState:
    print("   🔧 Editor: редактирую...")
    
    response = llm.invoke([
        SystemMessage(content="Улучши текст: исправь ошибки, сделай более читаемым."),
        HumanMessage(content=state["draft"])
    ])
    
    return {"final_text": response.content}

# Собираем subgraph
writing_workflow = StateGraph(WritingState)
writing_workflow.add_node("draft", draft_node)
writing_workflow.add_node("edit", edit_node)
writing_workflow.set_entry_point("draft")
writing_workflow.add_edge("draft", "edit")
writing_workflow.add_edge("edit", END)

writing_subgraph = writing_workflow.compile()

print("Writing Subgraph готов!")

In [ ]:
# === MAIN GRAPH: объединяем subgraphs ===

class MainState(TypedDict):
    user_query: str
    research_summary: str
    final_article: str

def call_research_team(state: MainState) -> MainState:
    print("\n📚 RESEARCH TEAM:")
    
    # Вызываем subgraph
    result = research_subgraph.invoke({
        "query": state["user_query"],
        "raw_data": "",
        "summary": ""
    })
    
    return {"research_summary": result["summary"]}

def call_writing_team(state: MainState) -> MainState:
    print("\n✍️ WRITING TEAM:")
    
    # Вызываем subgraph
    result = writing_subgraph.invoke({
        "source_material": state["research_summary"],
        "draft": "",
        "final_text": ""
    })
    
    return {"final_article": result["final_text"]}

# Собираем main graph
main_workflow = StateGraph(MainState)
main_workflow.add_node("research_team", call_research_team)
main_workflow.add_node("writing_team", call_writing_team)
main_workflow.set_entry_point("research_team")
main_workflow.add_edge("research_team", "writing_team")
main_workflow.add_edge("writing_team", END)

main_app = main_workflow.compile()

print("Main Graph с subgraphs готов!")

In [ ]:
# Запускаем!
result = main_app.invoke({
    "user_query": "Блокчейн в банковской сфере",
    "research_summary": "",
    "final_article": ""
})

print("\n" + "="*50)
print("ФИНАЛЬНАЯ СТАТЬЯ:")
print("="*50)
print(result["final_article"])

### Преимущества Subgraphs:

| Преимущество | Описание |
|--------------|----------|
| **Модульность** | Каждый subgraph — отдельный модуль |
| **Переиспользование** | Research team можно использовать в других проектах |
| **Тестирование** | Можно тестировать subgraph отдельно |
| **Читаемость** | Код организован логически |
| **Масштабирование** | Легко добавлять новые команды |

---

## 4. Практика: Полноценная команда агентов

Создадим реальную систему:
- **Planner** — разбивает задачу на шаги
- **Executor** — выполняет шаги
- **Reviewer** — проверяет результат

In [ ]:
# Состояние для Plan-Execute-Review
class PlanExecuteState(TypedDict):
    task: str
    plan: list[str]
    current_step: int
    results: Annotated[list, operator.add]
    final_result: str
    review: str
    approved: bool

print("State готов!")

In [ ]:
import json

def planner_node(state: PlanExecuteState) -> PlanExecuteState:
    print("\n📋 PLANNER: создаю план...")
    
    response = llm.invoke([
        SystemMessage(content="""Разбей задачу на 3-5 конкретных шагов.
Ответь JSON массивом строк: ["шаг 1", "шаг 2", ...]
Только JSON, без объяснений."""),
        HumanMessage(content=state["task"])
    ])
    
    try:
        # Пытаемся распарсить JSON
        content = response.content.strip()
        # Убираем markdown если есть
        if content.startswith("```"):
            content = content.split("```")[1]
            if content.startswith("json"):
                content = content[4:]
        plan = json.loads(content)
    except:
        # Fallback
        plan = ["Исследовать тему", "Написать черновик", "Отредактировать"]
    
    print(f"   План: {plan}")
    
    return {
        "plan": plan,
        "current_step": 0
    }

print("Planner создан!")

In [ ]:
def executor_node(state: PlanExecuteState) -> PlanExecuteState:
    step_idx = state["current_step"]
    current_step = state["plan"][step_idx]
    
    print(f"\n⚡ EXECUTOR: выполняю шаг {step_idx + 1}/{len(state['plan'])}")
    print(f"   Шаг: {current_step}")
    
    # Контекст предыдущих результатов
    prev_results = "\n".join(state.get("results", [])) if state.get("results") else "Нет"
    
    response = llm.invoke([
        SystemMessage(content=f"""Ты исполнитель. Выполни текущий шаг задачи.

Общая задача: {state['task']}
Текущий шаг: {current_step}
Предыдущие результаты: {prev_results}

Выполни шаг и верни результат. Кратко и по делу."""),
        HumanMessage(content=f"Выполни: {current_step}")
    ])
    
    result = f"[Шаг {step_idx + 1}] {current_step}: {response.content}"
    print(f"   Результат: {response.content[:100]}...")
    
    return {
        "results": [result],
        "current_step": step_idx + 1
    }

print("Executor создан!")

In [ ]:
def reviewer_node(state: PlanExecuteState) -> PlanExecuteState:
    print("\n🔍 REVIEWER: проверяю результаты...")
    
    all_results = "\n".join(state["results"])
    
    response = llm.invoke([
        SystemMessage(content="""Ты ревьюер. Проверь выполнение задачи.

Ответь в формате:
ОЦЕНКА: (1-10)
ВЕРДИКТ: APPROVED или NEEDS_REVISION
КОММЕНТАРИЙ: (краткий фидбек)"""),
        HumanMessage(content=f"""Задача: {state['task']}

Результаты:
{all_results}""")
    ])
    
    review_text = response.content
    approved = "APPROVED" in review_text.upper()
    
    print(f"   Вердикт: {'✅ APPROVED' if approved else '❌ NEEDS_REVISION'}")
    
    return {
        "review": review_text,
        "approved": approved,
        "final_result": all_results if approved else ""
    }

print("Reviewer создан!")

In [ ]:
# Роутеры
def should_continue_execution(state: PlanExecuteState) -> str:
    """Проверяем, есть ли ещё шаги."""
    if state["current_step"] < len(state["plan"]):
        return "executor"  # Ещё есть шаги
    return "reviewer"  # Все шаги выполнены

def after_review(state: PlanExecuteState) -> str:
    """После ревью — заканчиваем или переделываем."""
    if state["approved"]:
        return "__end__"
    # Можно вернуться к planner для переделки
    # Но для простоты — заканчиваем
    return "__end__"

print("Роутеры готовы!")

In [ ]:
# Собираем граф
per_workflow = StateGraph(PlanExecuteState)

# Узлы
per_workflow.add_node("planner", planner_node)
per_workflow.add_node("executor", executor_node)
per_workflow.add_node("reviewer", reviewer_node)

# Рёбра
per_workflow.set_entry_point("planner")
per_workflow.add_edge("planner", "executor")

per_workflow.add_conditional_edges(
    "executor",
    should_continue_execution,
    {
        "executor": "executor",
        "reviewer": "reviewer"
    }
)

per_workflow.add_conditional_edges(
    "reviewer",
    after_review,
    {
        "__end__": END
    }
)

# Компилируем с checkpointer
checkpointer = MemorySaver()
per_app = per_workflow.compile(checkpointer=checkpointer)

print("Plan-Execute-Review граф готов!")

In [ ]:
# Визуализация
try:
    display(Image(per_app.get_graph().draw_mermaid_png()))
except:
    print(per_app.get_graph().draw_mermaid())

In [ ]:
# Запускаем!
config = {"configurable": {"thread_id": "per-demo-1"}}

result = per_app.invoke({
    "task": "Создай краткий гайд по Docker для Java-разработчика",
    "plan": [],
    "current_step": 0,
    "results": [],
    "final_result": "",
    "review": "",
    "approved": False
}, config=config)

print("\n" + "="*50)
print("ФИНАЛЬНЫЕ РЕЗУЛЬТАТЫ:")
print("="*50)
print("\n📝 РЕЗУЛЬТАТЫ ВЫПОЛНЕНИЯ:")
for r in result["results"]:
    print(f"\n{r}")
print("\n" + "-"*50)
print("📋 РЕВЬЮ:")
print(result["review"])

---

## 5. Map-Reduce (Send API) — Параллельная обработка

### Идея:

Динамически создаём N параллельных "воркеров" через `Send()`. Каждый обрабатывает свою часть задачи, результаты автоматически собираются через reducer.

```
Query → Router → Send("worker", item1)
              → Send("worker", item2)  → Aggregator → Result
              → Send("worker", item3)
```

### Java аналогия:

**Fork-Join Framework / CompletableFuture.allOf():**
```java
List<CompletableFuture<String>> futures = items.stream()
    .map(item -> CompletableFuture.supplyAsync(() -> analyze(item)))
    .toList();
CompletableFuture.allOf(futures.toArray(new CompletableFuture[0])).join();
```

В LangGraph `Send()` — аналог `fork()`: создаёт N параллельных веток.
`Annotated[list, operator.add]` — аналог `join()`: собирает результаты воедино.

In [ ]:
# === Паттерн 5: Map-Reduce (Send API) ===

# Входные данные для отдельного воркера (приходят через Send)
class WorkerInput(TypedDict):
    subject: str       # Конкретный элемент для анализа
    topic: str         # Общий контекст

# Состояние графа
class MapReduceState(TypedDict):
    topic: str                                    # Общая тема анализа
    subjects: list[str]                           # Элементы для параллельной обработки
    results: Annotated[list, operator.add]        # Reducer: автоматически объединяет результаты
    final_report: str                             # Итоговый синтез

print("MapReduce State создан!")

In [ ]:
# Fan-out: создаём по одному Send на каждый subject
def mr_route_to_workers(state: MapReduceState) -> list[Send]:
    print(f"\n🗺️ ROUTER: распределяю {len(state['subjects'])} задач параллельно...")

    sends = []
    for subject in state["subjects"]:
        print(f"   → Send('mr_worker', '{subject}')")
        sends.append(Send("mr_worker", {
            "subject": subject,
            "topic": state["topic"]
        }))

    return sends

print("Router (Send API) создан!")

In [ ]:
# Worker: обрабатывает один элемент
def mr_worker(state: WorkerInput) -> dict:
    subject = state["subject"]
    topic = state["topic"]
    print(f"\n⚙️ WORKER: анализирую '{subject}'...")

    response = llm.invoke([
        SystemMessage(content=f"Ты аналитик технологий. Контекст: {topic}."),
        HumanMessage(content=f"Проанализируй {subject}: ключевые особенности, плюсы и минусы. Кратко, 3-4 предложения.")
    ])

    result = f"**{subject}**: {response.content}"
    print(f"   Готово: {result[:80]}...")

    return {"results": [result]}

print("Worker создан!")

In [ ]:
# Aggregator: синтезирует все результаты воркеров
def mr_aggregator(state: MapReduceState) -> dict:
    print(f"\n📊 AGGREGATOR: синтезирую {len(state['results'])} результатов...")

    all_results = "\n\n".join(state["results"])

    response = llm.invoke([
        SystemMessage(content="Ты аналитик. Сравни технологии и дай рекомендации: какую когда использовать. Кратко, структурированно."),
        HumanMessage(content=f"Тема: {state['topic']}\n\nАнализы:\n{all_results}")
    ])

    return {"final_report": response.content}

print("Aggregator создан!")

In [ ]:
# Собираем Map-Reduce граф
mr_workflow = StateGraph(MapReduceState)

# Узлы
mr_workflow.add_node("mr_worker", mr_worker)
mr_workflow.add_node("mr_aggregator", mr_aggregator)

# Fan-out: START → Send("mr_worker", ...) для каждого subject
mr_workflow.add_conditional_edges(START, mr_route_to_workers, ["mr_worker"])

# Fan-in: после ВСЕХ воркеров → aggregator
mr_workflow.add_edge("mr_worker", "mr_aggregator")
mr_workflow.add_edge("mr_aggregator", END)

mr_app = mr_workflow.compile()

print("Map-Reduce граф собран!")

# Визуализация
try:
    display(Image(mr_app.get_graph().draw_mermaid_png()))
except:
    print(mr_app.get_graph().draw_mermaid())

In [ ]:
# Запускаем Map-Reduce!
mr_result = mr_app.invoke({
    "topic": "Системы обмена сообщениями для микросервисов",
    "subjects": ["Kafka", "RabbitMQ", "Redis Streams"],
    "results": [],
    "final_report": ""
})

print("\n" + "="*50)
print("📋 СВОДНЫЙ ОТЧЁТ:")
print("="*50)
print(mr_result["final_report"])

### Что произошло:

1. **Router** получил 3 темы и создал 3 `Send()` — по одному на каждую технологию
2. **Worker** запустился 3 раза параллельно (каждый анализировал свою технологию)
3. `operator.add` **reducer** автоматически собрал результаты в один список
4. **Aggregator** синтезировал все анализы в сводный отчёт

### Ключевые моменты:

| Элемент | Роль |
|---------|------|
| `Send("node", state)` | Создаёт параллельную "ветку" выполнения |
| `Annotated[list, operator.add]` | Автоматически объединяет результаты из веток |
| Router → list[Send] | Динамическое количество воркеров |
| Worker → Aggregator | Fan-in: ждём завершения всех веток |

---

## 6. Reflection (Critique Loop) — Самоулучшение

### Идея:

Агент создаёт решение, критик его проверяет, агент улучшает — и так до одобрения или лимита итераций.

```
Task → Generator → Critic → APPROVED? → Result
                     ↓ нет
                  Generator → Critic → ...  (до 3 итераций)
```

### Java аналогия:

**CI Pipeline / Code Review Cycle:**
```java
// build → test → fix → repeat
while (!approved && iteration < MAX_RETRIES) {
    artifact = build(source);
    testResult = runTests(artifact);
    if (testResult.passed()) {
        approved = true;
    } else {
        source = fix(source, testResult.getErrors());
        iteration++;
    }
}
```

### Когда использовать:
- Юридические документы (LegalBPM!)
- Генерация кода
- Эссе и тексты
- Любая задача где качество критично

In [ ]:
# === Паттерн 6: Reflection (Critique Loop) ===

class ReflectionState(TypedDict):
    task: str              # Исходная задача
    draft: str             # Текущая версия решения
    critique: str          # Критика от ревьюера
    iteration: int         # Счётчик итераций
    is_satisfied: bool     # Флаг: критик одобрил?

print("Reflection State создан!")

In [ ]:
# Generator: создаёт/улучшает решение
def ref_generator(state: ReflectionState) -> dict:
    iteration = state.get("iteration", 0)

    if state.get("critique") and state.get("draft"):
        # Повторная итерация — улучшаем на основе критики
        print(f"\n✍️ GENERATOR (итерация {iteration + 1}): улучшаю решение...")
        prompt = f"Улучши решение на основе критики.\n\nПредыдущая версия:\n{state['draft']}\n\nКритика:\n{state['critique']}\n\nИсправь замечания и верни улучшенную версию."
    else:
        # Первая итерация
        print(f"\n✍️ GENERATOR (итерация 1): создаю решение...")
        prompt = state["task"]

    response = llm.invoke([
        SystemMessage(content="Ты опытный разработчик. Создай качественное решение. Пиши кратко и точно."),
        HumanMessage(content=prompt)
    ])

    print(f"   Решение: {response.content[:100]}...")

    return {"draft": response.content}

print("Generator создан!")

In [ ]:
# Critic: проверяет решение
def ref_critic(state: ReflectionState) -> dict:
    iteration = state.get("iteration", 0) + 1
    print(f"\n🧐 CRITIC (итерация {iteration}): проверяю решение...")

    response = llm.invoke([
        SystemMessage(content='''Ты строгий код-ревьюер. Проверь решение:
1. Корректность
2. Полнота
3. Читаемость

Если всё хорошо — начни ответ со слова APPROVED.
Если есть замечания — опиши их конкретно.'''),
        HumanMessage(content=f"Задача: {state['task']}\n\nРешение:\n{state['draft']}")
    ])

    is_approved = "APPROVED" in response.content.upper()
    verdict = "✅ APPROVED" if is_approved else "❌ NEEDS IMPROVEMENT"
    print(f"   Вердикт: {verdict}")
    if not is_approved:
        print(f"   Критика: {response.content[:100]}...")

    return {
        "critique": response.content,
        "is_satisfied": is_approved,
        "iteration": iteration
    }

print("Critic создан!")

In [ ]:
# Router: цикл до 3 итераций или APPROVED
def ref_router(state: ReflectionState) -> str:
    if state.get("is_satisfied"):
        print("\n✅ Критик одобрил! Завершаем.")
        return "__end__"
    if state.get("iteration", 0) >= 3:
        print("\n⏰ Лимит итераций (3). Завершаем с текущим результатом.")
        return "__end__"
    print(f"\n🔄 Итерация {state.get('iteration', 0)}/3 — отправляем на доработку...")
    return "generator"

print("Router создан!")

In [ ]:
# Собираем Reflection граф
ref_workflow = StateGraph(ReflectionState)

# Узлы
ref_workflow.add_node("generator", ref_generator)
ref_workflow.add_node("critic", ref_critic)

# Рёбра
ref_workflow.add_edge(START, "generator")
ref_workflow.add_edge("generator", "critic")

# Условный переход: цикл или конец
ref_workflow.add_conditional_edges(
    "critic",
    ref_router,
    {
        "generator": "generator",
        "__end__": END
    }
)

ref_app = ref_workflow.compile()

print("Reflection граф собран!")

# Визуализация
try:
    display(Image(ref_app.get_graph().draw_mermaid_png()))
except:
    print(ref_app.get_graph().draw_mermaid())

In [ ]:
# Запускаем Reflection!
ref_result = ref_app.invoke({
    "task": "Напиши SQL-запрос: отчёт по продажам за последний квартал с группировкой по категориям товаров, включая итоговую сумму и средний чек.",
    "draft": "",
    "critique": "",
    "iteration": 0,
    "is_satisfied": False
})

print("\n" + "="*50)
print("📝 ФИНАЛЬНОЕ РЕШЕНИЕ:")
print("="*50)
print(ref_result["draft"])
print("\n" + "-"*50)
print(f"🔄 Итераций: {ref_result['iteration']}")
print(f"✅ Одобрено: {ref_result['is_satisfied']}")

### Что произошло:

1. **Generator** создал первую версию SQL-запроса
2. **Critic** проверил — нашёл замечания (или одобрил сразу)
3. Если не одобрено — **Generator** улучшил решение на основе критики
4. Цикл повторяется до **APPROVED** или **3 итераций**

### Ключевые моменты:

| Элемент | Роль |
|---------|------|
| `iteration` counter | Защита от бесконечного цикла |
| `is_satisfied` flag | Критик управляет завершением |
| Conditional edge | Цикл generator ↔ critic |
| Critique в state | Контекст для улучшения |

### Когда это полезно в LegalBPM:
- Генерация юридических документов → проверка на соответствие шаблону
- Анализ договоров → итеративное уточнение выводов

---

## 7. Hierarchical Multi-Agent — Иерархия команд

### Идея:

Несколько уровней управления: Top Supervisor координирует команды, каждая команда — отдельный subgraph со своими агентами.

```
                    Top Supervisor
                   /              \
          Research Team        Content Team
          /         \          /         \
      Searcher   Analyst    Writer    Editor
```

### Java аналогия:

**Микросервисы с API Gateway → Domain Services → Workers:**
```java
// API Gateway (Top Supervisor)
@RestController
class GatewayController {
    @Autowired ResearchService researchService;  // Team 1
    @Autowired ContentService contentService;    // Team 2

    @PostMapping("/process")
    Result process(Request req) {
        ResearchResult research = researchService.analyze(req);  // Team делает свою работу
        ContentResult content = contentService.create(research); // Другая Team
        return combine(research, content);
    }
}
```

### Отличие от Supervisor (секция 2):
- Supervisor управляет **отдельными агентами** (плоская структура)
- Hierarchical управляет **командами** (каждая команда — subgraph с несколькими агентами)

In [ ]:
# === Паттерн 7: Hierarchical Multi-Agent ===

# Состояние Research Team
class HResearchState(TypedDict):
    task: str
    search_results: str
    analysis: str

# Состояние Content Team
class HContentState(TypedDict):
    brief: str
    draft: str
    edited_text: str

# Состояние главного графа
class HierarchicalState(TypedDict):
    task: str
    research_result: str
    content_result: str
    current_team: str
    final_output: str

print("Hierarchical States созданы!")

In [ ]:
# === Research Team (subgraph) ===

def h_searcher(state: HResearchState) -> dict:
    print("   🔎 Searcher: ищу информацию...")
    response = llm.invoke([
        SystemMessage(content="Ты исследователь. Найди 5 ключевых фактов по теме. Кратко, по пунктам."),
        HumanMessage(content=state["task"])
    ])
    return {"search_results": response.content}

def h_analyst(state: HResearchState) -> dict:
    print("   📊 Analyst: анализирую данные...")
    response = llm.invoke([
        SystemMessage(content="Ты аналитик. На основе фактов сделай структурированный анализ: тренды, выводы, рекомендации. Кратко."),
        HumanMessage(content=state["search_results"])
    ])
    return {"analysis": response.content}

# Собираем Research Team subgraph
h_research_wf = StateGraph(HResearchState)
h_research_wf.add_node("searcher", h_searcher)
h_research_wf.add_node("analyst", h_analyst)
h_research_wf.set_entry_point("searcher")
h_research_wf.add_edge("searcher", "analyst")
h_research_wf.add_edge("analyst", END)
h_research_team = h_research_wf.compile()

print("Research Team subgraph готов!")

In [ ]:
# === Content Team (subgraph) ===

def h_writer(state: HContentState) -> dict:
    print("   ✏️ Writer: пишу черновик...")
    response = llm.invoke([
        SystemMessage(content="Ты технический писатель. Напиши структурированную статью (3-4 абзаца) на основе брифа."),
        HumanMessage(content=state["brief"])
    ])
    return {"draft": response.content}

def h_editor(state: HContentState) -> dict:
    print("   🔧 Editor: редактирую текст...")
    response = llm.invoke([
        SystemMessage(content="Ты редактор. Улучши текст: структура, читаемость, точность формулировок. Верни финальную версию."),
        HumanMessage(content=state["draft"])
    ])
    return {"edited_text": response.content}

# Собираем Content Team subgraph
h_content_wf = StateGraph(HContentState)
h_content_wf.add_node("writer", h_writer)
h_content_wf.add_node("editor", h_editor)
h_content_wf.set_entry_point("writer")
h_content_wf.add_edge("writer", "editor")
h_content_wf.add_edge("editor", END)
h_content_team = h_content_wf.compile()

print("Content Team subgraph готов!")

In [ ]:
# === Top Supervisor + вызов команд ===

def h_top_supervisor(state: HierarchicalState) -> dict:
    print("\n👑 TOP SUPERVISOR думает...")

    context = f"Исследование: {'готово' if state.get('research_result') else 'не начато'}\nКонтент: {'готов' if state.get('content_result') else 'не начат'}"

    response = llm.invoke([
        SystemMessage(content='''Ты главный координатор двух команд:
- research_team: исследует тему, собирает и анализирует факты
- content_team: пишет и редактирует контент на основе исследования

Правила:
1. Сначала research_team (если исследование не готово)
2. Потом content_team (если контент не готов)
3. FINISH если всё готово

Ответь ОДНИМ словом: research_team, content_team или FINISH'''),
        HumanMessage(content=context)
    ])

    decision = response.content.strip().lower()
    if decision not in ["research_team", "content_team", "finish"]:
        if not state.get("research_result"):
            decision = "research_team"
        elif not state.get("content_result"):
            decision = "content_team"
        else:
            decision = "finish"

    print(f"   Решение: {decision}")
    return {"current_team": decision}

def h_call_research(state: HierarchicalState) -> dict:
    print("\n📚 RESEARCH TEAM:")
    result = h_research_team.invoke({
        "task": state["task"],
        "search_results": "",
        "analysis": ""
    })
    return {"research_result": result["analysis"]}

def h_call_content(state: HierarchicalState) -> dict:
    print("\n✍️ CONTENT TEAM:")
    result = h_content_team.invoke({
        "brief": f"Задача: {state['task']}\n\nИсследование:\n{state['research_result']}",
        "draft": "",
        "edited_text": ""
    })
    return {"content_result": result["edited_text"]}

def h_finalizer(state: HierarchicalState) -> dict:
    print("\n🏁 FINALIZER: формирую итог...")
    return {"final_output": state["content_result"]}

print("Top Supervisor и функции команд созданы!")

In [ ]:
# Роутер от Top Supervisor
def h_route(state: HierarchicalState) -> str:
    team = state.get("current_team", "research_team")
    if team == "finish":
        return "finalizer"
    elif team == "content_team":
        return "call_content"
    return "call_research"

# Собираем главный граф
h_workflow = StateGraph(HierarchicalState)

# Узлы
h_workflow.add_node("top_supervisor", h_top_supervisor)
h_workflow.add_node("call_research", h_call_research)
h_workflow.add_node("call_content", h_call_content)
h_workflow.add_node("finalizer", h_finalizer)

# Рёбра
h_workflow.set_entry_point("top_supervisor")

h_workflow.add_conditional_edges(
    "top_supervisor",
    h_route,
    {
        "call_research": "call_research",
        "call_content": "call_content",
        "finalizer": "finalizer"
    }
)

# После каждой команды — обратно к supervisor
h_workflow.add_edge("call_research", "top_supervisor")
h_workflow.add_edge("call_content", "top_supervisor")
h_workflow.add_edge("finalizer", END)

h_app = h_workflow.compile()

print("Hierarchical граф собран!")

# Визуализация
try:
    display(Image(h_app.get_graph().draw_mermaid_png()))
except:
    print(h_app.get_graph().draw_mermaid())

In [ ]:
# Запускаем Hierarchical!
h_result = h_app.invoke({
    "task": "Подготовь отчёт: микросервисы vs монолит для enterprise-проекта",
    "research_result": "",
    "content_result": "",
    "current_team": "",
    "final_output": ""
})

print("\n" + "="*50)
print("📄 ФИНАЛЬНЫЙ ОТЧЁТ:")
print("="*50)
print(h_result["final_output"])

### Что произошло:

1. **Top Supervisor** решил вызвать Research Team (исследование не готово)
2. **Research Team** внутри себя: Searcher → Analyst
3. Top Supervisor получил результат, вызвал **Content Team**
4. **Content Team** внутри себя: Writer → Editor
5. Top Supervisor увидел что всё готово → **Finalizer**

### Архитектура:

```
Top Supervisor (LLM-решения)
├── Research Team (subgraph)
│   ├── Searcher (поиск фактов)
│   └── Analyst (анализ)
├── Content Team (subgraph)
│   ├── Writer (черновик)
│   └── Editor (редактура)
└── Finalizer (итоговый результат)
```

### Когда использовать Hierarchical:

| Критерий | Supervisor | Hierarchical |
|----------|-----------|--------------|
| Количество агентов | 3-5 | 6+ |
| Структура | Плоская | Многоуровневая |
| Сложность задач | Средняя | Высокая |
| Пример | Чат-бот | Enterprise система |

---

## Итоги

### Что изучили:

| # | Паттерн | Когда использовать | Сложность |
|---|---------|-------------------|------------|
| 1 | **Handoff** | Простая цепочка агентов (A → B → C) | ⭐ |
| 2 | **Supervisor** | Динамическая маршрутизация, один "босс" | ⭐⭐ |
| 3 | **Subgraphs** | Модульная архитектура, переиспользование | ⭐⭐⭐ |
| 4 | **Plan-Execute-Review** | Сложные многошаговые задачи с проверкой | ⭐⭐⭐ |
| 5 | **Map-Reduce (Send)** | Параллельная обработка N элементов | ⭐⭐⭐ |
| 6 | **Reflection** | Итеративное улучшение (код, документы) | ⭐⭐ |
| 7 | **Hierarchical** | Большие команды с несколькими уровнями | ⭐⭐⭐⭐ |

### Java аналогии:

| LangGraph | Java/Spring |
|-----------|-------------|
| Supervisor | Orchestrator / Saga Coordinator |
| Workers | Microservices |
| Subgraphs | Maven Modules / Bounded Contexts |
| State | Saga State / Event Store |
| Send (Map-Reduce) | Fork-Join / CompletableFuture.allOf() |
| Reflection Loop | CI Pipeline (build → test → fix → repeat) |
| Hierarchical | API Gateway → Domain Services → Workers |

### Когда что использовать:

- **1 агент** — простые задачи, мало инструментов
- **Handoff** — фиксированный pipeline (A → B → C)
- **Supervisor** — динамический выбор одного агента из нескольких
- **Subgraphs** — модульность, переиспользование команд
- **Plan-Execute** — задачи с планированием и проверкой качества
- **Map-Reduce** — параллельная обработка N однотипных элементов (анализ, генерация)
- **Reflection** — итеративное улучшение (юр. документы, код, эссе)
- **Hierarchical** — большие системы с несколькими командами и уровнями управления

### Следующие шаги:

1. **LangGraph Cloud** — деплой в продакшен
2. **Память** — long-term memory для агентов
3. **Инструменты** — интеграция с реальными API
4. **Streaming** — потоковая обработка результатов
5. **Human-in-the-Loop** — взаимодействие с пользователем в процессе работы